<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.2-transient-heat/Ex08.2_05_compare_and_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.2 · Notebook 05 — Compare, and Report

**Paired with L8.2 · Dynamic Heat**

One equation, four models: a soft initial condition, a hard one, a domain with
a hole, and a diffusivity that was not given to you.

---

## 0 · Setup and what the other notebooks produced

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.2-transient-heat/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import course_core as cc
cc.keep_outputs("Ex08.2_outputs")


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
runs = {
    "01 · soft IC":       "nb01_soft.npz",
    "02 · hard IC":       "nb02_hard.npz",
    "03 · plate in time": "nb03_plate.npz",
    "04 · inverse alpha": "nb04_inverse.npz",
}
R = {}
cc.needed('nb01_soft.npz', 'nb02_hard.npz', 'nb03_plate.npz', 'nb04_inverse.npz')   # on Colab without Drive, asks for the missing files
for label, fn in runs.items():
    p = os.path.join(cc.OUTPUT_DIR, fn)
    if os.path.exists(p):
        R[label] = np.load(p, allow_pickle=True)
        print(f"  loaded  {label}")
    else:
        print(f"  MISSING {label}  ({fn}) -- run that notebook first")

## 0b · Your personal seed

The notebooks fix the seed to 88 so the "what you should see" blocks are true
on any machine. The report asks for numbers from **your** seed instead, so a
report cannot be copied between groups without the numbers giving it away.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own collocation draw through the slab, and what the exact solution does
# on it.
your_pts = pb.plate_spacetime_points(3000, seed=SEED)
T_you = pb.exact_transient(your_pts[:, 0], your_pts[:, 1], your_pts[:, 2])

print()
print(f"  your mean temperature over the slab : {T_you.mean():.6f}")
print(f"  your peak temperature               : {T_you.max():.6f}")
print(f"  fraction of your points with t < tau: "
      f"{(your_pts[:, 2] < pb.time_constant()).mean() * 100:.1f}%")

## 1 · The evidence, gathered

In [ ]:
if "01 · soft IC" in R and "02 · hard IC" in R:
    s, h = R["01 · soft IC"], R["02 · hard IC"]
    print("SOFT versus HARD initial condition")
    print(error_table(
        [[f"{t:.1f}", f"{a:.3e}", f"{b:.3e}", f"{c:.3e}", f"{d:.3e}"]
         for t, a, b, c, d in zip(s["ts"], s["rel"], h["rel"],
                                  s["abs_err"], h["abs_err"])],
        ["t", "soft rel", "hard rel", "soft abs", "hard abs"]))
    print(f"\n  hard IC, error at t = 0 before training : {float(h['ic_err']):.2e}")
    print(f"  hard IC, edge error before training     : {float(h['edge_err']):.2e}")
    print(f"  final loss, soft / hard : {s['lbfgs'][-1]:.3e} / "
          f"{h['lbfgs'][-1]:.3e}")

    plt.figure(figsize=(7.5, 3.4))
    plt.semilogy(s["ts"], s["abs_err"], "o-", label="soft IC")
    plt.semilogy(h["ts"], h["abs_err"], "s-", label="hard IC")
    plt.xlabel("t"); plt.ylabel("max absolute error")
    plt.legend(frameon=False); plt.grid(alpha=0.25, which="both")
    plt.tight_layout(); plt.show()

if "04 · inverse alpha" in R:
    inv = R["04 · inverse alpha"]
    print(f"\nINVERSE PROBLEM")
    print(f"  true / recovered alpha : {float(inv['alpha_true']):.4f} / "
          f"{float(inv['alpha_hat']):.4f}")
    print(f"  sensor window          : t = {float(inv['sensor_t'][0]):.2f} .. "
          f"{float(inv['sensor_t'][-1]):.2f}"
          f"   ({float(inv['sensor_t'][-1]) / pb.time_constant():.1f} tau)")

## 2 · Your answers

Replace every string. Keep to the word limits.

In [ ]:
# TODO: write your report. Every string below must be replaced.

Q1_EARLY_ERROR = """
(120 words) Why is the error largest at t = 0 with a soft initial condition?
Quote your own numbers from the soft/hard table above, and give at least two
distinct mechanisms -- one about the loss, one about the sampling.
(L8.2: the arrow of time is not in the loss)
"""

Q2_STEADY_LAST_FRAME = """
(120 words) Why does the steady state appear as the last frame of the
transient? Say what "last" has to mean for that to be true, using the time
constants in notebooks 00 and 03, and what you would see if you stopped too
early. (L8.2: the same plate, now transient)
"""

Q3_WHY_TRANSIENT = """
(150 words) Why identify alpha from a transient rather than from a steady
state? Answer in terms of what the steady equation does and does not contain.
Then report your recovered alpha and say how much the data supports it.
(L8.2: identifying diffusivity from a transient)
"""

Q4_HARMLESS_RELATIVE_ERROR = """
(120 words) Why can a relative error of 100% be harmless at late times? Use
the amplitude column from notebook 00 and your own absolute errors, and state
which of the two numbers you would put in an engineering report, and to whom.
(L8.2: pitfalls specific to transient heat)
"""

Q5_WHEN_CLASSICAL = """
(150 words) When does stiffness make you reach for a classical solver instead?
Name the property of the problem that decides it, not the tool.
(L8.2: stiffness; what a thermal engineer actually wants)
"""

Q6_SHORTER_CURVE = """
(150 words) What would happen to the inverse fit if the cooling curve were
shorter -- or, as in the notebook 04 question, moved later? Report what you
actually measured when you moved the sensor window, and give the general rule
for when a parameter is identifiable from data.
"""

NAME = "your name"
GROUP = "your group"

raise NotImplementedError("Write your report, then delete this line")

## 3 · Check, assemble, save

In [ ]:
answers = {
    "1 · Where the error goes": (Q1_EARLY_ERROR, 120),
    "2 · The steady state as the last frame": (Q2_STEADY_LAST_FRAME, 120),
    "3 · Identifying alpha from a transient": (Q3_WHY_TRANSIENT, 150),
    "4 · Relative against absolute": (Q4_HARMLESS_RELATIVE_ERROR, 120),
    "5 · When to use a classical solver": (Q5_WHEN_CLASSICAL, 150),
    "6 · A shorter cooling curve": (Q6_SHORTER_CURVE, 150),
}

problems = []
for title, (text, limit) in answers.items():
    words = len(text.split())
    if text.strip().startswith("(") or f"({limit} words)" in text:
        problems.append(f"{title}: still the prompt")
    elif words > limit * 1.15:
        problems.append(f"{title}: {words} words, limit {limit}")
    elif words < limit * 0.4:
        problems.append(f"{title}: {words} words, too short")

if problems:
    print("Not ready:")
    for p in problems:
        print("  -", p)
else:
    lines = ["# Ex_08.2 — Dynamic heat: a plate, a hole, and time", "",
             f"**{NAME}** · {GROUP}", "",
             f"study number {STUDENT_NUMBER} · seed {SEED}", "",
             "Deep Learning for Engineering · Aalborg University · 2026", "",
             "---", ""]
    for title, (text, _) in answers.items():
        lines += [f"## {title}", "", text.strip(), ""]
    out = os.path.join(cc.OUTPUT_DIR, "Ex08.2_report.md")
    with open(out, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines))
    print("wrote", out)
    print(f"{sum(len(t.split()) for t, _ in answers.values())} words total")

<!-- notebook-questions v1 -->
---

## The questions from notebooks 01 to 04

Every notebook in this set ended with four questions under *Before you move
on*. Copy your answers to them into the cell below — a few sentences each —
and the cell after it adds all 17, each under its question, to the end of the
report you just wrote. The last one is not from a notebook: it is the
question across all of them, and it concludes the report. On Colab every
notebook runs on its own machine, so this
notebook cannot read what you wrote in the others: copying is the only way
across.

Keep the answers short and in your own words. The arrow after each question
names the question on the lecture's Questions slide that it helps answer, so
this section is also your preparation for the oral examination.

In [ ]:
# notebook-questions v1 -- your answers from notebooks 01 to 04 ----------
# Paste each answer between its triple quotes. The question is in the
# comment above it; an empty answer is reported as not answered.

NOTEBOOK_ANSWERS = {

    # ---- notebook 01 · Soft Initial Condition ------------------------------------
    # 01.1 The initial condition is one term among three and carries about a
    # tenth of the points. Which of the three recorded terms fell first, and
    # which plateaued? Explain the ordering using the fact that the loss treats
    # a point at $t = 0.01$ exactly as it treats one at $t = 0.99$. (-> L8.2
    # Q5)
    "01.1": """
""",
    # 01.2 Which instant in section 5 has the largest error, and why is it that
    # one? In a time history at the plate centre, what would that error look
    # like at $t = 0$? Would doubling the interior points fix it? Say why or
    # why not. (-> L8.2 Q5, Q8)
    "01.2": """
""",
    # 01.3 The relative error at late times is large and the absolute error is
    # tiny. Say which of the two you would put in a report, and to whom. Then
    # read notebook 00's amplitude column: what does it say about where
    # $t_{\mathrm{end}}$ was put, and where most of the points were spent? (->
    # L8.2 Q8)
    "01.3": """
""",
    # 01.4 A time-marching solver would have started from the initial field
    # exactly and never lost it. Say what this network does with time instead,
    # what that buys you (section 6 slices the model at any instant you like),
    # and what it costs, using your own error at $t = 0$. (-> L8.2 Q2)
    "01.4": """
""",

    # ---- notebook 02 · Hard-Enforced Initial Condition ---------------------------
    # 02.1 The untrained network already satisfied both conditions. Say
    # precisely why, term by term in the trial solution. Then say what the
    # $(1-t)$ factor assumes about the time axis and about $f_{\mathrm{IC}}$ at
    # the edges, and what would break if $t_{\mathrm{end}}$ were 10 in unscaled
    # units. (-> L8.2 Q3)
    "02.1": """
""",
    # 02.2 The gap between soft and hard narrowed with time. Give the
    # mechanism. Explain why removing the initial-condition term is the first
    # remedy for a loss that does not know which way time runs, and what an
    # offset at $t = 0$ in notebook 01's time history meant. (-> L8.2 Q3, Q5)
    "02.2": """
""",
    # 02.3 This construction needed $f_{\mathrm{IC}}$ as a **formula**.
    # Describe what you would do if the initial field were a measured thermal
    # image, and what new error you would introduce. The image will not be
    # exactly zero at the edges: which assumption of the trial solution does
    # that break? (-> L8.2 Q3)
    "02.3": """
""",
    # 02.4 The benchmark decays as $e^{-2\pi^2 c t}$. Say what the diffusivity
    # $c$ sets here and what its units are in a physical problem. Notebook 00
    # printed two time constants, the slab's and the square's: say which one
    # belongs to this problem and why, how many of them the window
    # $t_{\mathrm{end}} = 1$ spans, and whether you would shorten it. (-> L8.2
    # Q1, Q8)
    "02.4": """
""",

    # ---- notebook 03 · The Plate with a Hole, in Time ----------------------------
    # 03.1 Has the field reached its steady state by $t = 0.5$? Quote the peak
    # temperatures you printed, not an impression. Section 4 puts this plate's
    # time constant near 0.13, longer than both of notebook 00's: say why a
    # plate that loses heat only through its hole settles more slowly than one
    # held at zero on its edges, and what you would change if the peak were
    # still rising. (-> L8.2 Q6, Q8)
    "03.1": """
""",
    # 03.2 The trial solution $t\,\phi\,\mathcal{N}$ has no
    # $(1-t)f_{\mathrm{IC}}$ term. Say why it can drop it here, and what it
    # still assumes about the range of $t$. Then list the conditions it
    # enforces exactly and the one left to a loss term, and say why the
    # insulated edges could not have been built into the trial solution the way
    # the hole was. (-> L8.2 Q6, Q3)
    "03.2": """
""",
    # 03.3 There is no exact solution to score against. Name two checks you can
    # still run. One should be the flux balance, `pb.flux_balance`, and the
    # other a time history at the hottest point. Say when the flux balance
    # applies in a transient, and what the history must do at $t = 0$ with this
    # trial solution. (-> L8.2 Q7)
    "03.3": """
""",
    # 03.4 The source is constant. What would you change if $Q$ switched on and
    # off with a duty cycle, and what would you expect the peak temperature to
    # do? Say what a PINN offers here that a time-marching solver does not, if
    # the on-time were made an extra input to the network. (-> L8.2 Q2)
    "03.4": """
""",

    # ---- notebook 04 · Recovering the Diffusivity --------------------------------
    # 04.1 Move the sensor times to `np.linspace(0.6, 1.0, 12)` and retrain.
    # Report the $\alpha$ you get, and explain it using notebook 00's amplitude
    # table and the noise level of 0.002. Why does the early transient identify
    # the diffusivity when the late one does not? How would you tell, without
    # knowing the truth, that a recovered value is unconstrained? (-> L8.2 Q10)
    "04.1": """
""",
    # 04.2 The notebook optimises $\log\alpha$ rather than $\alpha$. Say what a
    # negative $\alpha$ would mean physically. Then say what $\alpha$ sets in
    # this problem and what its units are, and point to the feature of the
    # cooling curve in section 3 that it controls. (-> L8.2 Q9)
    "04.2": """
""",
    # 04.3 The sensors read from $t = 0.01$ to $0.25$. Express that window in
    # time constants of the square, $\tau = L^2/(2\pi^2\alpha)$, for the true
    # $\alpha$. If you were designing the experiment, where would you end the
    # sensor window and the collocation window, and why do points far beyond
    # that teach the network little about $\alpha$? (-> L8.2 Q8, Q10)
    "04.3": """
""",
    # 04.4 Here $\alpha$ became one more trainable parameter, found in one
    # training run with the field. A time-marching solver would have to be
    # wrapped in an outer optimisation loop to do the same. Say what the PINN
    # does differently that makes this possible, and what it costs you. (->
    # L8.2 Q9)
    "04.4": """
""",

    # ---- to conclude, across all the notebooks ------------------------------
    # C One loop produced the soft start, the hard start, the plate and the
    # recovered diffusivity. What did time add to the loss, and which of your
    # results would convince someone that the network is right?
    # (-> L8.2, Exercise slide)
    "C": """
""",
}


In [ ]:
# notebook-questions v1 -- add the questions and your answers to the report ----------
# Safe to run again: it replaces the section rather than adding a second copy.
import os

NOTEBOOK_QUESTIONS = {
    "01.1": ('01', 'Soft Initial Condition', 'The initial condition is one term among three and carries about a tenth of the points. Which of the three recorded terms fell first, and which plateaued? Explain the ordering using the fact that the loss treats a point at $t = 0.01$ exactly as it treats one at $t = 0.99$.', 'L8.2 Q5'),
    "01.2": ('01', 'Soft Initial Condition', 'Which instant in section 5 has the largest error, and why is it that one? In a time history at the plate centre, what would that error look like at $t = 0$? Would doubling the interior points fix it? Say why or why not.', 'L8.2 Q5, Q8'),
    "01.3": ('01', 'Soft Initial Condition', "The relative error at late times is large and the absolute error is tiny. Say which of the two you would put in a report, and to whom. Then read notebook 00's amplitude column: what does it say about where $t_{\\mathrm{end}}$ was put, and where most of the points were spent?", 'L8.2 Q8'),
    "01.4": ('01', 'Soft Initial Condition', 'A time-marching solver would have started from the initial field exactly and never lost it. Say what this network does with time instead, what that buys you (section 6 slices the model at any instant you like), and what it costs, using your own error at $t = 0$.', 'L8.2 Q2'),
    "02.1": ('02', 'Hard-Enforced Initial Condition', 'The untrained network already satisfied both conditions. Say precisely why, term by term in the trial solution. Then say what the $(1-t)$ factor assumes about the time axis and about $f_{\\mathrm{IC}}$ at the edges, and what would break if $t_{\\mathrm{end}}$ were 10 in unscaled units.', 'L8.2 Q3'),
    "02.2": ('02', 'Hard-Enforced Initial Condition', "The gap between soft and hard narrowed with time. Give the mechanism. Explain why removing the initial-condition term is the first remedy for a loss that does not know which way time runs, and what an offset at $t = 0$ in notebook 01's time history meant.", 'L8.2 Q3, Q5'),
    "02.3": ('02', 'Hard-Enforced Initial Condition', 'This construction needed $f_{\\mathrm{IC}}$ as a **formula**. Describe what you would do if the initial field were a measured thermal image, and what new error you would introduce. The image will not be exactly zero at the edges: which assumption of the trial solution does that break?', 'L8.2 Q3'),
    "02.4": ('02', 'Hard-Enforced Initial Condition', "The benchmark decays as $e^{-2\\pi^2 c t}$. Say what the diffusivity $c$ sets here and what its units are in a physical problem. Notebook 00 printed two time constants, the slab's and the square's: say which one belongs to this problem and why, how many of them the window $t_{\\mathrm{end}} = 1$ spans, and whether you would shorten it.", 'L8.2 Q1, Q8'),
    "03.1": ('03', 'The Plate with a Hole, in Time', "Has the field reached its steady state by $t = 0.5$? Quote the peak temperatures you printed, not an impression. Section 4 puts this plate's time constant near 0.13, longer than both of notebook 00's: say why a plate that loses heat only through its hole settles more slowly than one held at zero on its edges, and what you would change if the peak were still rising.", 'L8.2 Q6, Q8'),
    "03.2": ('03', 'The Plate with a Hole, in Time', 'The trial solution $t\\,\\phi\\,\\mathcal{N}$ has no $(1-t)f_{\\mathrm{IC}}$ term. Say why it can drop it here, and what it still assumes about the range of $t$. Then list the conditions it enforces exactly and the one left to a loss term, and say why the insulated edges could not have been built into the trial solution the way the hole was.', 'L8.2 Q6, Q3'),
    "03.3": ('03', 'The Plate with a Hole, in Time', 'There is no exact solution to score against. Name two checks you can still run. One should be the flux balance, `pb.flux_balance`, and the other a time history at the hottest point. Say when the flux balance applies in a transient, and what the history must do at $t = 0$ with this trial solution.', 'L8.2 Q7'),
    "03.4": ('03', 'The Plate with a Hole, in Time', 'The source is constant. What would you change if $Q$ switched on and off with a duty cycle, and what would you expect the peak temperature to do? Say what a PINN offers here that a time-marching solver does not, if the on-time were made an extra input to the network.', 'L8.2 Q2'),
    "04.1": ('04', 'Recovering the Diffusivity', "Move the sensor times to `np.linspace(0.6, 1.0, 12)` and retrain. Report the $\\alpha$ you get, and explain it using notebook 00's amplitude table and the noise level of 0.002. Why does the early transient identify the diffusivity when the late one does not? How would you tell, without knowing the truth, that a recovered value is unconstrained?", 'L8.2 Q10'),
    "04.2": ('04', 'Recovering the Diffusivity', 'The notebook optimises $\\log\\alpha$ rather than $\\alpha$. Say what a negative $\\alpha$ would mean physically. Then say what $\\alpha$ sets in this problem and what its units are, and point to the feature of the cooling curve in section 3 that it controls.', 'L8.2 Q9'),
    "04.3": ('04', 'Recovering the Diffusivity', 'The sensors read from $t = 0.01$ to $0.25$. Express that window in time constants of the square, $\\tau = L^2/(2\\pi^2\\alpha)$, for the true $\\alpha$. If you were designing the experiment, where would you end the sensor window and the collocation window, and why do points far beyond that teach the network little about $\\alpha$?', 'L8.2 Q8, Q10'),
    "04.4": ('04', 'Recovering the Diffusivity', 'Here $\\alpha$ became one more trainable parameter, found in one training run with the field. A time-marching solver would have to be wrapped in an outer optimisation loop to do the same. Say what the PINN does differently that makes this possible, and what it costs you.', 'L8.2 Q9'),
    "C": ('C', 'to conclude', 'One loop produced the soft start, the hard start, the plate and the recovered diffusivity. What did time add to the loss, and which of your results would convince someone that the network is right?', 'L8.2, Exercise slide'),
}

report_md = os.path.join(cc.OUTPUT_DIR, "Ex08.2_report.md")
HEAD = "## Questions from the notebooks"
if not os.path.exists(report_md):
    print("No Ex08.2_report.md yet: run the cell that writes the report first.")
else:
    text = open(report_md, encoding="utf-8").read()
    text = text.split("\n" + HEAD)[0].rstrip() + "\n"
    out = ["", HEAD, "",
           "Each question is tagged with the lecture question it serves.", ""]
    missing, current = [], None
    for key, (nb, title, question, ref) in NOTEBOOK_QUESTIONS.items():
        if nb != current:
            out += ["### To conclude" if nb == "C" else f"### Notebook {nb} · {title}", ""]
            current = nb
        answer = NOTEBOOK_ANSWERS.get(key, "").strip()
        if not answer:
            missing.append(key)
        out += [f"**{key}.** {question} *(→ {ref})*", "",
                answer or "*not answered*", ""]
    with open(report_md, "w", encoding="utf-8") as fh:
        fh.write(text + "\n".join(out))
    print(f"added to {report_md}: {len(NOTEBOOK_QUESTIONS) - len(missing)} of "
          f"{len(NOTEBOOK_QUESTIONS)} answered")
    if missing:
        print("not answered:", ", ".join(missing))


**What you should see.** `added to .../Ex08.2_report.md: 17 of 17 answered`.
Until then it lists the questions still empty, and the report says *not
answered* under each of them — to the marker too. Run this cell again after any
change to the report above it, because rewriting the report removes the
section; then make the PDF.

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex08.2_report.md into Ex08.2_report.pdf, with any figure
# saved as Ex08.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex08.2_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open(os.path.join(cc.OUTPUT_DIR, "Ex08.2_report.md"), encoding="utf-8").read()
figs = sorted(glob.glob("Ex08.2_report*.png")
              + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex08.2_report*.png")))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download(pdf_path)
except ImportError:
    pass


## 4 · Extensions

Optional, and each one is a short experiment rather than a new exercise.

- Give the source a duty cycle, `Q(t) = Q0 if (t*4)%1 < 0.55 else 0`, and find
  the peak temperature. Does it exceed the steady value at full load?
- Halve `t_end` and re-run. Which conclusions change?
- Implement causal weighting from L8.2's four remedies and compare with hard enforcement.
- Two materials with diffusivities differing by 100×. Where does it break?

## 5 · What Ex_08.2 was for

Three claims you can now defend with your own measurements:

* **Where you put known information decides how much of it survives.** The
  initial condition in the loss is a request the optimiser may refuse; the same
  condition inside the trial solution is a fact it cannot.
* **A transient is not uniformly hard, and one error number hides that.** You
  reported error against time, relative *and* absolute, because at late times
  one of the two is meaningless.
* **An optimiser will always return a parameter.** Whether the data supports it
  is a separate question, and one you have to ask yourself.

From here the geometry and the physics both get harder, and the exact solution
you have been checking against mostly disappears.